<a href="https://colab.research.google.com/github/Kepha-M/financial-inclusion-africa/blob/main/Zindi_FinancialInclusion_Module1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestClassifier

train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')

train.head()


,country,year,uniqueid,bank_account,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
0,Kenya,2018,uniqueid_1,Yes,Rural,Yes,3,24,Female,Spouse,Married/Living together,Secondary education,Self employed
1,Kenya,2018,uniqueid_2,No,Rural,No,5,70,Female,Head of Household,Widowed,No formal education,Government Dependent
2,Kenya,2018,uniqueid_3,Yes,Urban,Yes,5,26,Male,Other relative,Single/Never Married,Vocational/Specialised training,Self employed
3,Kenya,2018,uniqueid_4,No,Rural,Yes,5,34,Female,Head of Household,Married/Living together,Primary education,Formally employed Private
4,Kenya,2018,uniqueid_5,No,Urban,No,8,26,Male,Child,Single/Never Married,Primary education,Informally employed


In [ ]:
# Target variable
y = (train['bank_account'] == 'Yes').astype(int)
X = train.drop(columns=['bank_account'])

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print('Training rows:', len(train))
print('Test rows:', len(test))


Training rows: 23524
Test rows: 10086


In [ ]:
# Preprocessing
preprocess = ColumnTransformer([
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ]), num_cols)
])


In [ ]:
# Model
model = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring='neg_mean_absolute_error'
)

print('CV MAE:', -scores.mean())


CV MAE: 0.1697395844629031


In [ ]:
# Train final model
model.fit(X, y)

predictions = model.predict(test)

submission = pd.DataFrame({
    'unique_id': test['uniqueid'].astype(str) + ' x ' + test['country'].astype(str),
    'bank_account': predictions
})

submission.to_csv('submission.csv', index=False)

submission.head()


,unique_id,bank_account
0,uniqueid_6056 x Kenya,1
1,uniqueid_6060 x Kenya,1
2,uniqueid_6065 x Kenya,0
3,uniqueid_6072 x Kenya,0
4,uniqueid_6073 x Kenya,0
